In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — MONTAGE ET FICHE D'UN JOUEUR : STAVELOT B – HONSFELD B, CAMÉRA MANUELLE (sans refaire l'analyse) — 25/11/2026
# ⚠️ Les DEUX équipes jouent en rouge (Stavelot bordeaux / short bleu marine, Honsfeld rouge / short noir) : la couleur
#   seule ne les sépare pas. Le plus sûr pour ce premier essai : un GARDIEN (maillot d'une autre couleur), ou un joueur
#   dont le numéro est bien lisible. Couleurs reconnues : rouge, bleu, bleu clair, blanc, noir, jaune, vert, orange,
#   violet, gris (bordeaux → « rouge »).
# ENTRÉES (Add Input) :
#   • la vidéo (dataset matchcomplet : match_testcomplet.mp4) ;
#   • la SORTIE du notebook de l'analyse du match (Notebook Output) : cache_testcomplet_reference.zip,
#     bout_en_bout_testcomplet.json, gemini_cache_testcomplet.jsonl, rapport_testcomplet*.pdf ;
#   • codeprojectai (parseq_scoutia_maillots_v2.pt) ; mziparameter (param.env, nouvelle clé Gemini).
# GitHub à jour : analysis/montage_joueur.py, analysis/montage_joueur_script.py, export/fiche_joueur.py,
#   export/rapport_client.py, export/resume_match.py.
# ACCÉLÉRATEUR : GPU T4 conseillé — les numéros de ce match n'ont jamais été lus (pas de fichier de lectures) : la lecture
#   PARSeq se fait ici (≈ 1 h 30 à 2 h 30 sans GPU, estimation). Coût Gemini estimé ≈ 0,25 à 0,35 $. Save & Run All.
# À GARDER : joueur_testcomplet_<numéro>_<couleur>.zip (montage, fiche, séquences pour « Composer son montage », lectures des
#   numéros réutilisables pour un autre joueur du même match).
# ════════════════════════════════════════════════════════════════════════════════════════════
MATCH = "testcomplet"
# ← À REMPLIR : numéro et couleur de MAILLOT de l'équipe ; gardien : ajouter gardien=True et la couleur DE SON maillot,
#   ex. dict(numero=1, couleur="bleu clair", gardien=True)
# n° 11 de STAVELOT : maillot bordeaux (→ « rouge »), SHORT BLEU MARINE (→ « bleu ») — le short départage les deux
# équipes en rouge (Honsfeld : short noir). Nécessite analysis/montage_joueur*.py du 25/11 (paramètre short)
JOUEURS = [dict(numero=11, couleur="rouge", short="bleu")]

import glob, json, os, shutil, subprocess, sys, zipfile

REPO = "/kaggle/working/ProjectAIMika"
if not os.path.isdir(REPO):
    if os.system(f"git clone -q --depth 1 https://github.com/mikarocco-droid/ProjectAIMika.git {REPO}") != 0:
        raise SystemExit("❌ clone du dépôt impossible")
for f_ in ("analysis/montage_joueur.py", "analysis/montage_joueur_script.py", "export/fiche_joueur.py"):
    if not os.path.exists(f"{REPO}/{f_}"):
        raise SystemExit(f"❌ {f_} absent de GitHub")
if "MESURE_PAR_SHORT" not in open(f"{REPO}/analysis/montage_joueur_script.py", encoding="utf-8").read():
    raise SystemExit("❌ analysis/montage_joueur_script.py du 25/11 (couleur du short) absent de GitHub")
if "COMPOSER SON MONTAGE" not in open(f"{REPO}/analysis/montage_joueur_script.py", encoding="utf-8").read():
    print("⚠️ analysis/montage_joueur_script.py n'est pas la dernière version (pas de « Composer son montage ») : "
          "mettez-la sur GitHub avant de lancer.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "pytorch-lightning", "nltk", "fpdf2", "pypdf",
                "google-genai", "opencv-python-headless"], check=False)
sys.path.insert(0, REPO)


def _un(motif, quoi, obligatoire=True):
    r = sorted(glob.glob(f"/kaggle/input/**/{motif}", recursive=True))
    if not r and obligatoire:
        raise SystemExit(f"❌ {quoi} introuvable dans les entrées")
    return r[0] if r else None


video = _un("match_testcomplet.mp4", "la vidéo du match (matchcomplet)")
relais = json.load(open(_un(f"bout_en_bout_{MATCH}.json", f"bout_en_bout_{MATCH}.json"), encoding="utf-8"))
_un(f"gemini_cache_{MATCH}*.jsonl", "les réponses Gemini de l'analyse du match")
poids = _un("parseq_scoutia_maillots_v2.pt", "parseq_scoutia_maillots_v2.pt (codeprojectai)")
rapport = next((r for r in sorted(glob.glob(f"/kaggle/input/**/rapport_{MATCH}*.pdf", recursive=True)) if "_joueur" not in r), None)

# cache du suivi : archive zip → dossier
cache = f"/kaggle/working/cache_{MATCH}"
# 25/11 : Kaggle DÉZIPPE les archives des datasets → on accepte aussi un DOSSIER cache_<match>_reference*/frames_data.pkl
_dos_c = next(iter(sorted(glob.glob(f"/kaggle/input/**/cache_{MATCH}_reference*/**/frames_data.pkl", recursive=True)
                         + glob.glob(f"/kaggle/input/**/cache_{MATCH}_reference*/frames_data.pkl", recursive=True))), None)
if _dos_c and not os.path.exists(f"{cache}/frames_data.pkl"):
    cache = os.path.dirname(_dos_c)
if not os.path.exists(f"{cache}/frames_data.pkl"):
    cz = _un(f"cache_{MATCH}_reference*.zip", f"cache_{MATCH}_reference.zip (ou son dossier dézippé)")
    os.makedirs(cache, exist_ok=True)
    with zipfile.ZipFile(cz) as z:
        for n in z.namelist():
            if n.endswith("/"):
                continue
            with z.open(n) as src, open(os.path.join(cache, os.path.basename(n)), "wb") as dst:
                shutil.copyfileobj(src, dst)
print(f"cache du suivi : {cache} ({', '.join(sorted(os.listdir(cache)))})")

cle = ""
pe = _un("mziparameter/**/param.env", "param.env (mziparameter)", obligatoire=False) or _un("param.env", "param.env", False)
for l in open(pe, encoding="utf-8") if pe else []:
    if l.strip().startswith("GEMINI_API_KEY="):
        cle = l.split("=", 1)[1].strip().strip('"').strip("'")
if not cle:
    raise SystemExit("❌ clé Gemini absente de param.env")

import torch
print(f"GPU : {'oui (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'NON — lecture des numéros plus lente'}")

from analysis.montage_joueur import monter_joueur

for j in JOUEURS:
    nom = f"joueur_{MATCH}_{j['numero']}_{j['couleur'].replace(' ', '_')}" + (f"_short_{j['short']}" if j.get("short") else "")
    sortie = f"/kaggle/working/{nom}"
    print(f"\n{'═' * 90}\nJOUEUR n°{j['numero']} {j['couleur']}{' (gardien)' if j.get('gardien') else ''} — {MATCH}\n{'═' * 90}", flush=True)
    res = monter_joueur(video=video, cache_dir=cache, sortie=sortie, match=MATCH, bornes=relais.get("bornes"),
                        buts=relais.get("buts"), entrees=["/kaggle/input", "/kaggle/working"], gemini_key=cle,
                        parseq_poids=poids, rapport_pdf=rapport, **j)
    for k in ("trouve", "erreur", "message", "presence_s", "passages", "moments_ballon", "buts_confirmes",
              "buts_possibles", "buts_impliques", "arrets", "tirs_adverses", "cout_gemini_usd", "duree_min", "montage",
              "fiche", "fiche_donnees", "sequences_joueur", "rapport_et_fiche", "journal"):
        if k in res:
            print(f"   {k:18s} : {res[k]}")
    shutil.make_archive(f"/kaggle/working/{nom}", "zip", "/kaggle/working", nom)
    print(f"\n✅ À télécharger : /kaggle/working/{nom}.zip")

print("\nPour la démonstration : dézippez l'archive, puis "
      "python outils/demo_resultat.py --dossier demo_stavelot --email … --titre \"Stavelot B vs Honsfeld B\" --joueur demo_stavelot\\joueur_testcomplet_…")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.6/360.6 kB 13.2 MB/s eta 0:00:00
cache du suivi : /kaggle/input/datasets/michaelziant/codeprojectai/cache_testcomplet_reference (events.pkl, frames_data.pkl, jersey_map.pkl, meta.json, team_colors.pkl)
GPU : oui (Tesla T4)

══════════════════════════════════════════════════════════════════════════════════════════
JOUEUR n°11 rouge — testcomplet
══════════════════════════════════════════════════════════════════════════════════════════
   relais : fourni par le pipeline
   BOUT EN BOUT : testcomplet | cache neuf /kaggle/input/datasets/michaelziant/codeprojectai/cache_testcomplet_reference | 0 action(s) du résumé produites par le pipeline
✅ Cache : /kaggle/input/datasets/michaelziant/codeprojectai/cache_testcomplet_reference
   coup d'envoi 00:10 | 34604 événements | 31740 images suivies | 0 numéros lus
✅ time_since_update présent (100.0 % des positions) : seules